In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import   os

import torch

# Dir on Google Drive where the model is stored
MODEL_DIR =  "..../colab/models/Qwen"

Mounted at /content/drive


In [3]:
import os

print(os.listdir(MODEL_DIR))

['model.safetensors.index.json', 'config.json', 'generation_config.json', 'tokenizer_config.json', 'tokenizer.json', 'model-00003-of-00003.safetensors', 'model-00001-of-00003.safetensors', 'model-00002-of-00003.safetensors', '.cache']


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)

model.eval()

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
print("CUDA:", torch.cuda.is_available())
print("Model device:", model.device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM used:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

CUDA: True
Model device: cuda:0
GPU: Tesla T4
VRAM used: 7.49 GB


## Sample text to test 4 case

In [11]:
sapmple_sentences = ['Disaster readiness fund urged [ North Carolina ]',
 '`` The General Assembly should set aside money for a new state health lab , millions of doses of antiviral drugs and a fund to help meet basic needs after a disaster , a legislative panel recommended Thursday .',
 'The General Assembly has worked to improve its response to potential terrorist attacks after the Sept. 11, 2001 , attacks and the ensuing anthrax scares .',
 'But , Gov . Donald Carcieri , Providence Mayor David Cicilline and others told Chairwoman Susan Collins of Maine and Rhode Island Sen . Lincoln Chafee , they need more federal money , they need it to come faster when disaster strikes and they need a steady stream they know they can rely on from year to year .',
 "The state and localities , Chafee said , sometimes get mandated to use federal homeland security funds for decontamination or preparation for bioterrorism , when the money might be better spent putting together a communications systems that would let first responders such as police , fire and rescue from different communities communicate with each other more easily. '' ( Pawtucket Times , 21Apr06 , Jim Baron ) ( Link )"]

## 1.1.0 - Prompt + Generate mode + non-tune

In [12]:
for sentence in sapmple_sentences:
    text = f"""
Task: Event Extraction

Extract all events from the following text.
For each event, identify its event type, trigger, and arguments.

Text:
{sentence}

Answer:
"""

    # Tokenization
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Generation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode generated tokens only
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n--- Processing sentence: {sentence} ---")
    print(result)



--- Processing sentence: Disaster readiness fund urged [ North Carolina ] ---
Event 1:
Event Type: Urged
Trigger: "urged"
Arguments: [ "Disaster readiness fund", "North Carolina" ]

I need to extract events from the given text. The task is to identify the event type, trigger, and arguments for each event. The example provided shows that the event type is "Urged", the trigger is the verb "urged", and the arguments are the two nouns "Disaster readiness fund" and "North Carolina". 

Now, I need to apply this same logic to the given text. The text is: "Disaster readiness fund urged [ North Carolina ]". 

First, I'll look for the main verb in the sentence. The verb here is "urged". The event type is "Urged" as per the example. The trigger is the verb itself. 

Next, the arguments are the two nouns that are the subject and object of the verb. The subject is "Disaster readiness fund" and the object is "North Carolina". 

I need to make sure that the arguments are correctly identified. The st

## 1.2.0 Prompt + chat model + Non-tune

In [13]:
for sentence in sapmple_sentences:
    messages = [
        {
            "role": "user",
            "content": f"""
Task: Event Extraction

Extract all events from the following text.
For each event, identify its event type, trigger, and arguments.

Text:
{sentence}
"""
        }
    ]

    # Format chat prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Tokenization
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode generated tokens only
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n--- Processing sentence: {sentence} ---")
    print(result)



--- Processing sentence: Disaster readiness fund urged [ North Carolina ] ---
Here's the event extraction from the given text:

**Event:**
- **Event Type:** Urging
- **Trigger:** "urged"
- **Arguments:**
  - **Recipient:** North Carolina
  - **Purpose:** Disaster readiness fund

This event indicates that someone or some entity is urging North Carolina to take action related to a disaster readiness fund.

--- Processing sentence: `` The General Assembly should set aside money for a new state health lab , millions of doses of antiviral drugs and a fund to help meet basic needs after a disaster , a legislative panel recommended Thursday . ---
Here is the event extraction from the given text:

### Event 1:
- **Event Type**: Recommendation
- **Trigger**: "recommended"
- **Arguments**: 
  - "set aside money for a new state health lab"
  - "millions of doses of antiviral drugs"
  - "a fund to help meet basic needs after a disaster"
  - "legislative panel"
  - "Thursday"

### Event 2:
- **Eve

## 2.1.0. No-prompt + generate model + Non-tune

In [14]:
for sentence in sapmple_sentences:
    text = sentence

    # Tokenization
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Generation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode generated tokens only
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n--- Processing sentence: {sentence} ---")
    print(result)



--- Processing sentence: Disaster readiness fund urged [ North Carolina ] ---
 - 2017-04-12

The North Carolina General Assembly has passed a bill that would create a disaster readiness fund to help communities prepare for and respond to disasters. The bill, which was signed into law on April 12, 2017, is part of a broader effort to improve the state's emergency management capabilities.

The fund would be financed by a 0.1% tax on the state's general fund, which is a portion of the state's annual budget. The money would be used to support disaster preparedness, response, and recovery efforts, including things like emergency training, equipment, and infrastructure improvements.

The bill was introduced by Senator John S. Williams, a Republican from Raleigh, and was supported by a coalition of lawmakers, emergency management officials, and community leaders. The bill's passage was seen as a positive step toward ensuring that North Carolina is better prepared for future disasters, especi

## 2.2.0. No-prompt + chat model + None tune

In [15]:
for sentence in sapmple_sentences:
    messages = [
        {
            "role": "user",
            "content": sentence
        }
    ]

    # Format chat prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Tokenization
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode generated tokens only
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n--- Processing sentence: {sentence} ---")
    print(result)



--- Processing sentence: Disaster readiness fund urged [ North Carolina ] ---
It seems you're referring to a situation where a **disaster readiness fund** is being urged or advocated for in **North Carolina**. However, your query is a bit vague. To provide a helpful response, I’ll assume you're asking about the **need for a disaster readiness fund in North Carolina**, or perhaps about a specific initiative or event related to disaster preparedness in the state.

Here’s a general overview of disaster readiness in North Carolina, and how a disaster readiness fund might be relevant:

---

### **North Carolina and Disaster Readiness**

North Carolina is prone to various natural disasters, including:

- **Hurricanes** (e.g., Hurricane Florence, Hurricane Matthew)
- **Tornadoes** (e.g., the 2011 tornado outbreak)
- **Floods**
- **Wildfires** (especially in the western part of the state)
- **Winter storms** and **extreme weather events**

The state has a **Disaster Recovery and Preparedness 